# Genre Popularity Trends

## Q1: Explode multi-genre strings

- Converts semi-structured genre data into analyzable rows; causes dataset expansion

- Explosion time + Peak memory usage


## Q2: Yearly average rating by genre

- Core trend metric; classic MapReduce pattern (GroupBy + aggregation)

- GroupBy time + Rows/sec


## Q3: Genre volume over time

- Tracks production trends per genre

- Count aggregation speed + Memory


## Q4: Weighted popularity score

- Improves metric quality using vote_count weighting

- CPU time + Column memory cost


## Q5: Top performing genre per year

- Requires ranking across groups

- Ranking time + Sorting overhead



In [1]:
# Import libraries
import pandas as pd
import numpy as np
import time
import tracemalloc
import ast
import sys

# Start Total Timer
start_total = time.perf_counter()
tracemalloc.start()

# Load Data
movies = pd.read_csv("movies_cleaned.csv")

print("Initial Shape:", movies.shape)
print("Initial Memory (MB):", round(movies.memory_usage(deep=True).sum() / 1024**2, 4))


# PREPARATION

# Ensure release_date is datetime
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")

# Create year column
movies["year"] = movies["release_date"].dt.year

# Ensure genres is list
def parse_list(x):
    if isinstance(x, str):
        try:
            if x.startswith("["):
                return ast.literal_eval(x)
            else:
                return [i.strip() for i in x.split(",")]
        except:
            return ["Unknown"]
    return x

movies["genres"] = movies["genres"].apply(parse_list)


# QUERY 1
start = time.perf_counter()
snapshot1 = tracemalloc.take_snapshot()

df_exploded = movies.explode("genres")

snapshot2 = tracemalloc.take_snapshot()
end = time.perf_counter()

q1_time = end - start
q1_memory = sum(stat.size_diff for stat in snapshot2.compare_to(snapshot1, 'lineno')) / 1024**2

print("\nQ1: Explode Genres")
print("Rows after explosion:", df_exploded.shape[0])
print("Explosion Time (sec):", round(q1_time, 4))
print("Memory Change (MB):", round(q1_memory, 4))


# QUERY 2
start = time.perf_counter()

grouped = df_exploded.groupby(["year","genres"])["vote_average"].mean().reset_index()

end = time.perf_counter()
q2_time = end - start
rows_per_sec = len(df_exploded) / q2_time

print("\nQ2: Yearly Avg Rating by Genre")
print("GroupBy Time (sec):", round(q2_time, 4))
print("Rows/sec:", round(rows_per_sec, 2))


# QUERY 3
start = time.perf_counter()
snapshot1 = tracemalloc.take_snapshot()

volume = df_exploded.groupby(["year","genres"]).size().reset_index(name="count")

snapshot2 = tracemalloc.take_snapshot()
end = time.perf_counter()

q3_time = end - start
q3_memory = sum(stat.size_diff for stat in snapshot2.compare_to(snapshot1, 'lineno')) / 1024**2

print("\nQ3: Genre Volume Over Time")
print("Count Aggregation Time (sec):", round(q3_time, 4))
print("Memory Change (MB):", round(q3_memory, 4))


# QUERY 4
start = time.perf_counter()

C = df_exploded["vote_average"].mean()
m = df_exploded["vote_count"].quantile(0.75)

df_exploded["weighted_score"] = (
    (df_exploded["vote_count"]/(df_exploded["vote_count"]+m)) * df_exploded["vote_average"] +
    (m/(df_exploded["vote_count"]+m)) * C
)

end = time.perf_counter()
q4_time = end - start
col_memory = df_exploded["weighted_score"].memory_usage(deep=True) / 1024**2

print("\nQ4: Weighted Popularity Score")
print("CPU Time (sec):", round(q4_time, 4))
print("Weighted Column Memory Cost (MB):", round(col_memory, 4))


# QUERY 5
start = time.perf_counter()

ranked = (
    grouped.sort_values(["year","vote_average"], ascending=[True,False])
           .groupby("year")
           .first()
           .reset_index()
)

end = time.perf_counter()
q5_time = end - start

print("\nQ5: Top Performing Genre Per Year")
print("Ranking + Sorting Time (sec):", round(q5_time, 4))


# TOTAL EXECUTION TIME
end_total = time.perf_counter()
total_time = end_total - start_total

print("\nTotal Execution Time (sec):", round(total_time, 4))

Initial Shape: (590202, 166)
Initial Memory (MB): 2350.4787

Q1: Explode Genres
Rows after explosion: 976441
Explosion Time (sec): 94.1053
Memory Change (MB): 626.5771

Q2: Yearly Avg Rating by Genre
GroupBy Time (sec): 1.6852
Rows/sec: 579438.56

Q3: Genre Volume Over Time
Count Aggregation Time (sec): 105.1829
Memory Change (MB): 0.0549

Q4: Weighted Popularity Score
CPU Time (sec): 0.415
Weighted Column Memory Cost (MB): 14.8993

Q5: Top Performing Genre Per Year
Ranking + Sorting Time (sec): 0.108

Total Execution Time (sec): 853.905
